<a href="https://colab.research.google.com/github/Krishay-Keshwani/Projects/blob/main/comment_catagory_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description about this project

The objective was to predict how an online discussion system categorizes user-generated comments into four distinct classes. The core challenge was handling a highly imbalanced, multimodal dataset—specifically, blending unstructured text with heavily skewed numerical engagement metrics and sparse categorical metadata.

Here is a breakdown of my methodology and technical approach:

1. Data Wrangling & Feature Engineering
Rather than feeding raw data into a model, I built a systematic feature engineering pipeline to extract meaningful signals:

Temporal & Meta Features: I parsed timestamps to engineer behavioral features (hour of posting, weekend flags) and calculated text meta-features (comment length, word count) to quantify user verbosity.

Taming Skewed Distributions: Engagement metrics (like upvotes and downvotes) contained massive, right-skewed outliers. I applied a logarithmic (log1p) transformation to compress these outliers, stabilizing the variance for downstream models.

Text Normalization: I developed a custom regex pipeline to strip URLs, remove non-alphanumeric artifacts, and normalize whitespace, ensuring a clean corpus before vectorization.

2. Robust Preprocessing Architecture
To prevent data leakage and ensure reproducibility, I designed a strict Scikit-Learn Pipeline utilizing a ColumnTransformer to handle diverse data types concurrently:

Numerical & Categorical: Handled missing values via mean/most-frequent imputation, scaled numericals using StandardScaler, and applied OneHotEncoder to categorical flags.

Text Vectorization: Deployed a TfidfVectorizer capped at 25,000 features. Crucially, I included bi-grams (ngram_range=(1,2)) to capture context and used sublinear_tf=True to apply logarithmic scaling to term frequencies, preventing highly repetitive words from dominating the model.

3. Iterative Modeling & Optimization
I treated this as a rigorous scientific experiment, benchmarking complexity iteratively:

Dimensionality Reduction: For memory-intensive models (like Random Forests and Multi-Layer Perceptrons), I utilized TruncatedSVD to compress the massive TF-IDF sparse matrix down to 300 principal components, retaining critical variance while drastically cutting training time.

Hyperparameter Tuning: I heavily utilized RandomizedSearchCV, explicitly optimizing for the Macro F1 Score to ensure the minority classes weren't ignored during training.

Handling Imbalance: Across all baseline models (including Logistic Regression and neural networks), I combated the severe class imbalance (Class 0 at 57% vs. Class 3 at <3%) by leveraging balanced class weights and tuning regularization parameters.

4. The Champion Model: LightGBM
My winning iteration abandoned SVD compression. Instead, I combined the dense tabular features with the 25,000-dimensional TF-IDF sparse matrix using scipy.sparse.hstack, feeding it directly into a Light Gradient Boosting Machine (LGBMClassifier).

Because LightGBM is natively optimized for high-dimensional sparse data, I was able to fine-tune its hyperparameters for maximum generalization. I increased n_estimators to 1200 with a slow learning_rate of 0.03, and crucially, utilized colsample_bytree=0.6. By forcing each tree to evaluate only 60% of the features at a time, I effectively applied feature dropout, forcing the ensemble to learn from subtle textual cues rather than overfitting on dominant words.

Results:
This strategy successfully balanced the massive feature space with the severe class imbalance, resulting in a robust validation accuracy of ~91.3% and a peak Macro F1 score of 0.813.

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

comment_category_prediction_challenge_path = kagglehub.competition_download('comment-category-prediction-challenge')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")
sampleSub = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")

In [ ]:
train.describe()

In [1]:
train.head()

NameError: name 'train' is not defined

In [ ]:
test.describe()

In [ ]:
test.shape

NameError: name 'test' is not defined

In [ ]:
test.head()

In [ ]:
train['label'].unique()

In [ ]:
train['comment'].unique()

In [ ]:
from sklearn.dummy import DummyClassifier as DC

In [ ]:
X = train.drop(columns = ['label'])
y = train["label"]

In [ ]:
train.shape

In [ ]:
X.shape

In [ ]:
dumdum = DC(strategy = 'prior')
dumdum.fit(X,y)

In [ ]:
pred = dumdum.predict(test)

In [ ]:
pred

In [ ]:
sampleSub['label'] = pred

In [ ]:
print("Columns with 'object' datatype (using df.dtypes):")
count = 0
for col in train.columns:
    if train[col].dtype == 'object':
        print(col)
        count+=1
print(count)

In [ ]:
count = 0
for dt in train.dtypes:
  print(dt)
  if dt == 'int64' or dt == 'bool':
    count+=1
count

In [ ]:
#sampleSub.to_csv("submission.csv",index=False)

In [ ]:
for i in train.columns:
  print(train[i].dtype, 'is the dt of = ',i)



In [ ]:
nan_columns = []
for col in train.columns:
    if train[col].isnull().any():
        nan_columns.append(col)

print("Columns with NaN values:", nan_columns)

print("\nNumber of NaN values per column:")
print(train[nan_columns].isnull().sum())

In [ ]:
print("Distinct target classes are:", len(train['label'].unique()))


In [ ]:
label_0_count = train['label'].value_counts().get(0, 0)
total_rows = len(train)


percentage_label_0 = (label_0_count / total_rows) * 100
print(label_0_count)
print(total_rows)
#percentage of 0
print(percentage_label_0)

In [ ]:
#percentage of all labels
label_percentage = train['label'].value_counts(normalize = True)*100
label_percentage

In [ ]:
median_upvotes = train['upvote'].median()
print(f"The median number of upvotes per comment is: {median_upvotes}")

In [ ]:
numerical_cols = train.select_dtypes(include=['int64', 'bool']).columns.tolist()

max_value = 0
max_col = 'aaaaaa'

for col in numerical_cols:
    current_max = train[col].max()

    if current_max > max_value:
        max_value = current_max
        max_col = col

print("largest value column =", max_col)
print("largest value =", max_value)

In [ ]:
train['if_2'].min()

In [ ]:
#converting 'create_date' to datetime object in training set
train['created_date'] = pd.to_datetime(train['created_date'],errors='coerce')
train['created_date']

In [ ]:
train['created_date'].dt.month_name().mode()

In [ ]:
train.head()

In [ ]:
#Creating a total emoticon column which is the sum of all the emoticons and finding its max value
train['total_emoticon'] = train['emoticon_1']+train['emoticon_2']+train['emoticon_3']
train['total_emoticon']

In [ ]:
train['total_emoticon'].max()

In [ ]:
#replacing missing comments by empty string
train['comment'].fillna('')

In [ ]:
#finding all the rows where label has value 3 and then finding the median comment length
train_label_is3 = train[train['label'] == 3]
train_label_is3.head()

In [ ]:
comm_len = []

for i in train_label_is3['comment']:
    comm_len.append(len(i))
pd.Series(comm_len).median()

In [ ]:
#applying Min-Max scaling to upvotes and finding the scaled value of upvote value = 10
min_upv = train['upvote'].min()
max_upv = train['upvote'].max()

train['upvote_scaled'] = ( train['upvote'] - min_upv ) / ( min_upv + max_upv )
train[train['upvote'] == 10]['upvote_scaled']

In [ ]:
#filling all NaN values with empty string
train['comment'] = train['comment'].fillna('')

In [ ]:
#total number of words average length
words = train['comment'][train['label'] == 1].str.split().str.len()
avg_words = words.mean()
print("average word count for label 1 = ",avg_words)

In [ ]:
#count number of comments where Trump is imcluded
president = train['comment'].str.lower().str.contains("trump").sum()
president

In [ ]:
#taking the very first comment and removing certain words and punctuations
first_comm = train['comment'].head(1)[0]
import string
pun_lis = string.punctuation
stopwords = ['a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to', 'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his', 'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your']
for i in pun_lis: # removing punctuations
    first_comm = first_comm.replace(i,'')
f_comm_list = first_comm.lower().split()
for i in stopwords:
    for j in f_comm_list:
        if i == j :
            f_comm_list.remove(i)

            # removing stopwords

print(len(f_comm_list))

In [ ]:
#coverting all text in column to lowercase and tokenizing using space and hence calculating unique tokens

t_lower = train['comment'].astype(str).str.lower()
total_tokens = 0
unique_t = []
for i in t_lower:
    for j in i.split():
        unique_t.append(j)
print(len(set(unique_t)))

In [ ]:
#Appling the TfidfVectorizer to the comment column of train.csv with stop_words as "english", min_df as 5 and ngram_range as (1,2)

from sklearn.feature_extraction.text import TfidfVectorizer as TV
vc = TV(stop_words = 'english',
        min_df = 5,
        ngram_range = (1,2))
tfid_matrix = vc.fit_transform(train['comment'])

In [ ]:
tfid_matrix.shape[1] #features

In [ ]:
X = train.drop(columns = ['label'])
y = train["label"]

In [ ]:
#Spliting the training dataset into train and validation sets
from sklearn.model_selection import train_test_split

X_train,X_val,y_train,y_val = train_test_split(X,y,test_size = 0.2 , random_state = 42 , stratify = y)
print(X_train.shape ,'= shape of X_train')
print(X_val.shape ,'= shape of X_val')
print('a+c = ', X_train.shape[0]+X_val.shape[0])

In [ ]:
#finding the most frequent month
X_train['created_date'].dt.month_name().mode()

In [ ]:
#imputing all missing values with 'none'
X_catagorical = X_train.select_dtypes(include = 'object').columns.tolist()
for col in X_catagorical:
    X_train[col] = X_train[col].fillna('none')
    X_val[col] = X_val[col].fillna('none')
X_catagorical

In [ ]:
X_train['gender'].unique()

In [ ]:
#using one hot encoding on catagorical features except on 'comment'
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown = 'ignore',sparse_output = False)
Columns_for_encoding = ['race', 'religion', 'gender']
X_train_en = ohe.fit_transform(X_train[Columns_for_encoding])
X_val_en = ohe.transform(X_val[Columns_for_encoding])
t_ohe_en = pd.DataFrame(X_train_en,columns = ohe.get_feature_names_out(Columns_for_encoding),index = X_train.index)
v_ohe_en = pd.DataFrame(X_val_en,columns = ohe.get_feature_names_out(Columns_for_encoding),index = X_val.index)
X_train = pd.concat([X_train.drop(columns = Columns_for_encoding), t_ohe_en],axis = 1)
X_val = pd.concat([X_val.drop(columns = Columns_for_encoding), v_ohe_en],axis = 1)


In [ ]:
X_train.shape

In [ ]:
#vectorizing the comments in X train and val
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X_train_comm_en = vectorizer.fit_transform(X_train['comment'])
X_val_comm_en = vectorizer.transform(X_val['comment'])
X_train_comm_en

In [ ]:
X_train_comm_en[1, :].sum()

In [ ]:
#converting the disablity column to integer
X_train['disability'] = X_train['disability'].astype(int)
X_val['disability'] = X_val['disability'].astype(int)
print('sum of disability in X_train and X_val = ',X_train['disability'].sum()+X_val['disability'].sum())

In [ ]:
#extracting the day and hour from created date and then applying standard scaler to all numerical features
X_train['hour_of_day'] = X_train['created_date'].dt.hour.fillna(0)
X_val['hour_of_day'] = X_val['created_date'].dt.hour.fillna(0)

X_train['day_of_week'] = X_train['created_date'].dt.dayofweek.fillna(0)
X_val['day_of_week'] = X_val['created_date'].dt.dayofweek.fillna(0)

X_train = X_train.drop(columns=['created_date'])
X_val = X_val.drop(columns=['created_date'])
numerical_features = X_train.select_dtypes(include = ['int64','float64']).columns.tolist()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_val[numerical_features] = scaler.transform(X_val[numerical_features])
scaler.n_features_in_

In [ ]:
"""#applying TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['comment'])
X_val_tfidf = tfidf_vectorizer.transform(X_val['comment'])
X_train_features = X_train.drop(columns=['comment'])
X_val_features = X_val.drop(columns=['comment'])
X_train_sparse = csr_matrix(X_train_features.values)
X_val_sparse = csr_matrix(X_val_features.values)
X_train = hstack([X_train_sparse, X_train_tfidf])
X_val = hstack([X_val_sparse, X_val_tfidf])"""

In [ ]:
"""from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
mnb = MultinomialNB()
mnb.fit(X_train, y_train)
y_train_pred = mnb.predict(X_train)
macro_f1_train = f1_score(y_train, y_train_pred, average='macro')
print("Macro F1 Score on the training dataset:" ,macro_f1_train)"""

In [ ]:
"""import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

# 1. Load Data and Initial Split
# Ensure a clean start by reloading the data if 'train' DataFrame was modified in previous steps
train = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')

X = train.drop('label', axis=1)
y = train['label']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Feature Engineering for `created_date` and `disability` on X_train and X_val
# Converting 'created_date' to datetime and extract features
for df in [X_train, X_val]:
    df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
    df['day'] = df['created_date'].dt.day.fillna(0).astype(int)
    df['month'] = df['created_date'].dt.month.fillna(0).astype(int)
    df['year'] = df['created_date'].dt.year.fillna(0).astype(int)
    df['disability'] = df['disability'].astype(int)
    df.drop(columns=['created_date'], inplace=True)
    # Explicitly fill NaN values in the 'comment' column with empty strings for TfidfVectorizer
    df['comment'] = df['comment'].fillna('')


# Identify feature types for ColumnTransformer
# These are the original numerical columns, plus new date features, and 'disability'
numerical_features = [
    'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
    'upvote', 'downvote', 'if_1', 'if_2', 'day', 'month', 'year', 'disability'
]
categorical_features = ['race', 'religion', 'gender'] # These are still objects
text_feature = 'comment'


# 3. Define Preprocessing Pipelines for MultinomialNB
# Numerical Transformer: SimpleImputer (median) + MinMaxScaler
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()) # Ensures non-negative values for MultinomialNB
])

# Categorical Transformer: SimpleImputer (constant 'none') + OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='none')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Text Transformer: TfidfVectorizer
text_transformer = TfidfVectorizer(stop_words='english')


# 4. Build ColumnTransformer for Feature Integration
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features),
        ('text', text_transformer, text_feature)
    ],
    remainder='drop' # Explicitly drop columns not specified
)

# 5. Construct and Train Full Pipeline with MultinomialNB
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', MultinomialNB())
])

# Fit this complete pipeline on the X_train and y_train data.
model_pipeline.fit(X_train, y_train)

# 6. Evaluate Model on Training Data
# Use the trained pipeline to make predictions on the X_train dataset.
y_train_pred = model_pipeline.predict(X_train)

# Calculate and print the macro F1 score for these predictions against the true labels y_train.
macro_f1_train = f1_score(y_train, y_train_pred, average='macro')
#Evaluating on validation data
y_val_pred = model_pipeline.predict(X_val)
macro_f1_val = f1_score(y_val, y_val_pred, average='macro')
print(f"Macro F1 Score on the training dataset: {macro_f1_train}")
print(f"Macro F1 Score on the validation dataset: {macro_f1_val}")

# Final Task Output Summary
print(f"\nSummary of Data Preprocessing and Model Training:")
print(f"1. Data Loading and Initial Split: Loaded 'train.csv', split into 80% training and 20% validation sets (X_train, X_val, y_train, y_val) using stratified sampling.")
print(f"2. Feature Engineering: 'created_date' was converted to datetime, then 'day', 'month', and 'year' were extracted as new numerical features. The original 'created_date' column was dropped. The 'disability' column was converted to integer type (0 or 1).")
print(f"3. Preprocessing Pipelines Defined:")
print(f"   - Numerical Transformer: Applied SimpleImputer (median strategy) followed by MinMaxScaler to ensure non-negative values suitable for MultinomialNB.")
print(f"   - Categorical Transformer: Applied SimpleImputer (constant 'none') followed by OneHotEncoder (handling unknown categories and outputting dense arrays for clarity).")
print(f"   - Text Transformer: Used TfidfVectorizer (with English stop words) for the 'comment' column, generating sparse TF-IDF features.")
print(f"4. ColumnTransformer Built: A ColumnTransformer was used to concurrently apply these pipelines to their respective feature sets (numerical, categorical, text), combining them into a single feature matrix.")
print(f"5. Model Pipeline Constructed and Trained: A Pipeline was built, chaining the ColumnTransformer and a MultinomialNB classifier. This full pipeline was then fitted on the preprocessed X_train and y_train data.")
print(f"6. Evaluation: The macro F1 score on the training dataset was calculated as: {macro_f1_train}")
print(f"This strategy successfully addresses the challenges of mixed data types and the non-negativity assumption of MultinomialNB by using appropriate imputation, scaling (MinMaxScaler), encoding, and text vectorization techniques within a robust scikit-learn pipeline framework.")"""

In [ ]:
"""import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
# 1. Reload the `train.csv` dataset
train = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')

# 2. Create a feature DataFrame `X` by dropping the 'label' column from `train`
X = train.drop('label', axis=1)

# 3. Create a target Series `y` containing the 'label' column from `train`
y = train['label']

# 4. Split `X` and `y` into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
for df in [X_train, X_val]:
    df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
    df['day'] = df['created_date'].dt.day.fillna(0).astype(int)
    df['month'] = df['created_date'].dt.month.fillna(0).astype(int)
    df['year'] = df['created_date'].dt.year.fillna(0).astype(int)
    df['day_of_week'] = df['created_date'].dt.dayofweek.fillna(0).astype(int) # 0=Monday, 6=Sunday
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0) # Saturday and Sunday

    df['disability'] = df['disability'].astype(int)
    df['comment'] = df['comment'].fillna('')

    df.drop(columns=['created_date', 'day_of_week'], inplace=True)


# Identify feature types for ColumnTransformer
# These are the original numerical columns, plus new date features, and 'disability'
numerical_features = [
    'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
    'upvote', 'downvote', 'if_1', 'if_2', 'day', 'month', 'year', 'is_weekend', 'disability'
]
categorical_features = ['race', 'religion', 'gender'] # These are still objects
text_feature = 'comment'

# Numerical Transformer: SimpleImputer (median) + MinMaxScaler
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()) # Ensures non-negative values for MultinomialNB
])

# Categorical Transformer: SimpleImputer (constant 'none') + OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='none')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Text Transformer: TfidfVectorizer
text_transformer = TfidfVectorizer(stop_words='english')


preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features),
        ('text', text_transformer, text_feature)
    ],
    remainder='drop' # Explicitly drop columns not specified
)

# 5. Construct and Train Full Pipeline with MultinomialNB
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', MultinomialNB())
])

# Fit this complete pipeline on the X_train and y_train data.
model_pipeline.fit(X_train, y_train)

# 6. Evaluate Model on Training Data
# Use the trained pipeline to make predictions on the X_train dataset.
y_train_pred = model_pipeline.predict(X_train)

# Calculate and print the macro F1 score for these predictions against the true labels y_train.
macro_f1_train = f1_score(y_train, y_train_pred, average='macro')
#Evaluating on validation data
y_val_pred = model_pipeline.predict(X_val)
macro_f1_val = f1_score(y_val, y_val_pred, average='macro')
print(f"Macro F1 Score on the training dataset: {macro_f1_train}")
print(f"Macro F1 Score on the validation dataset: {macro_f1_val}")

# 1. Load the 'test.csv' file into a pandas DataFrame named test_df
test_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv', engine='python', on_bad_lines='warn')

# 2. Load the 'Sample.csv' file into a pandas DataFrame named sample_submission_df
sample_submission_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# 3. Print the shape of test_df and sample_submission_df
print(f"Shape of test_df: {test_df.shape}")
print(f"Shape of sample_submission_df: {sample_submission_df.shape}")

# Apply the same feature engineering and cleaning steps to test_df
test_df['created_date'] = pd.to_datetime(test_df['created_date'], errors='coerce')
test_df['day'] = test_df['created_date'].dt.day.fillna(0).astype(int)
test_df['month'] = test_df['created_date'].dt.month.fillna(0).astype(int)
test_df['year'] = test_df['created_date'].dt.year.fillna(0).astype(int)
test_df['day_of_week'] = test_df['created_date'].dt.dayofweek.fillna(0).astype(int) # 0=Monday, 6=Sunday
test_df['is_weekend'] = test_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0) # Saturday and Sunday

test_df['disability'] = test_df['disability'].astype(int)
test_df['comment'] = test_df['comment'].fillna('')

test_df.drop(columns=['created_date', 'day_of_week'], inplace=True)

print("First 5 rows of test_df after feature engineering and cleaning:")
display(test_df.head())
X_test_processed = model_pipeline.named_steps['preprocessor'].transform(test_df)

print(f"Shape of X_test_processed after preprocessing: {X_test_processed.shape}")
y_test_pred = model_pipeline.predict(test_df)

# Create the submission DataFrame
submission = pd.DataFrame({
    'ID': sample_submission_df['ID'],
    'label': y_test_pred
})

# Save the submission file
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully.")
print("First 5 rows of submission.csv:")
display(submission.head())"""

# milestone 4

**Q1)**
 Load the dataset and fill the NaN values in comment column with empty string then follow the below steps

A) Split the dataset after doing above step using train_test_split into 60% train and 40% validation. Keep random_state = 2306 and set stratify to the target variable (y).

After splitting, compute the class-wise counts of the training labels and validation labels using value_counts().

Before storing the counts, sort the class labels in ascending order to ensure consistent ordering. Store the resulting counts as NumPy arrays.

B) Load the same dataset and split it again using train_test_split, keeping all configurations the same as in Part A (60% train and 40% validation, random_state = 2306), but set stratify = None.

Again, compute the sorted class-wise counts of the training labels and validation labels and store them as NumPy arrays.

C) Validation Distribution Difference

Using the validation label counts from Part A (stratified) and Part B (non-stratified)
Convert both validation class count arrays into proportions by dividing each count by the total number of validation samples.

Make sure the class labels remain in ascending order (same order as before).

For each class, compute the absolute difference between the two proportions.


Find the maximum of these absolute differences.


In [ ]:
"""from sklearn.model_selection import train_test_split
train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
train['comment'] = train['comment'].fillna('')
X = train.drop(['label'],axis = 1)
y = train['label']
X_strain,y_strain,X_sval,y_sval = train_test_split(X,y,test_size = 0.4, random_state = 2306, stratify = y) # straitify = y values
X_nstrain,y_nstrain,X_nsval,y_nsval = train_test_split(X,y,test_size = 0.4, random_state = 2306, stratify = None) # straitify = None values
#calculating each of the values
y_sval_values = y_sval.value_counts().sort_index().to_numpy()
y_strain_values = y_strain.value_counts().sort_index().to_numpy()
y_nsval_values = y_nsval.value_counts().sort_index().to_numpy()
y_nstrain_values = y_nstrain.value_counts().sort_index().to_numpy()
ysval_proportions = y_sval_values / np.sum(y_sval_values)
ynsval_proportions = y_nsval_values / np.sum(y_nsval_values)
#difference
max_abs_diff = np.max(abs(ysval_proportions - ynsval_proportions))
max_abs_diff
"""

**Q2)**
 Use the stratified train–validation split obtained in Q1 (60% train, 40% validation, random_state=2306, stratify=y).

All preprocessing objects must be fit only on the training data and then used to transform both training and validation sets.

Step1: Column Removal
From both x_train and x_test:
Drop the column "created_date".

Step 2: Separate Text Column
Before any tabular preprocessing, Extract the "comment" column:
text_x_train = x_train["comment"]
text_x_test = x_test["comment"]
Remove "comment" from both x_train and x_test.

Step 3: Tabular Preprocessing
Apply a ColumnTransformer with the following exact configuration:
Handling categorical columns
Categorical columns are ["race", "religion", "gender", "disability"]

Process the above categorical columns using below techniques :
SimpleImputer(strategy="most_frequent")
OneHotEncoder(handle_unknown="ignore", sparse_output=True)
Handling numerical columns
numerical columns are ["post_id", "emoticon_1", "emoticon_2", "emoticon_3",
 "upvote", "downvote", "if_1", "if_2"]
Process the above numerical columns using below techniques:
SimpleImputer(strategy="mean")
StandardScaler() (default parameters)

ColumnTransformer Settings
remainder="passthrough"
Fit only on x_train
Transform both x_train and x_test
Store the transformed matrices as:
x_train_tabular
x_test_tabular


Step 4: Text Cleaning
Define the following function exactly:
def normalize_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', '  ', text).strip()
    return text

apply it using .apply() on both text_x_train and text_x_test then store the result as text_x_train_norm, text_x_test_norm

Step 5: TF-IDF Vectorization
Use: TfidfVectorizer(stop_words="english", max_features=5000)

Fit only on text_x_train_norm and Transform both train and test text
Store as: tf_idf_train, tf_idf_test
Do not change any other parameters.

Step 6: Combine Features

Use: hstack to combine features in the following exact order:

X_train_final = hstack([x_train_tabular, tf_idf_train])
X_test_final = hstack([x_test_tabular, tf_idf_test])
The result must remain a sparse matrix.

Give the sum of all values of X_train_final, enter the answer upto 3 decimal places.
Note: handle the null values of comment column (if any) by replacing them with empty string to avoid errors.

In [ ]:
"""from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.impute import SimpleImputer
from scipy.sparse import csr_matrix , hstack
import re
from sklearn.feature_extraction.text import TfidfVectorizer

train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
train['comment'] = train['comment'].fillna('')
X = train.drop(['label'],axis = 1)
y = train['label']
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size = 0.4, random_state = 2306, stratify = y) # straitify = y values
X_train = X_train.drop(columns=['created_date'],axis = 1)
X_val = X_val.drop(columns=['created_date'],axis = 1)
text_x_train = X_train["comment"]
text_x_val = X_val["comment"]
X_train = X_train.drop(columns=['comment'],axis = 1)
X_val = X_val.drop(columns=['comment'],axis = 1)
categorical_cols = ["race", "religion", "gender", "disability"]
numerical_cols = ["post_id", "emoticon_1", "emoticon_2", "emoticon_3",
                  "upvote", "downvote", "if_1", "if_2"
]
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])
preprocessor_tabular = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder="passthrough"
)
x_train_tabular = preprocessor_tabular.fit_transform(X_train)
x_test_tabular = preprocessor_tabular.transform(X_val)
def normalize_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
text_x_train_norm = text_x_train.apply(normalize_text)
text_x_test_norm = text_x_val.apply(normalize_text)
tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
tf_idf_train = tfidf_vectorizer.fit_transform(text_x_train_norm)
tf_idf_test = tfidf_vectorizer.transform(text_x_test_norm)
X_train_final = hstack([x_train_tabular, tf_idf_train])
X_test_final = hstack([x_test_tabular, tf_idf_test])
sum_X_train_final = X_train_final.sum()

print("Sum of X train final sparse matrix = ",sum_X_train_final)



"""

**prerequisites**
For all subsequent questions (Q3 onwards), reduce the feature space using TruncatedSVD(n_components=300, random_state=2306)

Instructions:
Fit the TruncatedSVD only on X_train_final.
Transform both  X_train_final and X_test_final
Store the transformed matrices as X_train_reduced and X_test_reduced.

Use X_train_reduced for all model training in the remaining questions.Do not refit SVD for each question. Do not change n_components.

In [ ]:
"""from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components = 300, random_state = 2306)
svd.fit(X_train_final)
X_train_reduced = svd.transform(X_train_final)
X_test_reduced = svd.transform(X_test_final)
print("the shape of reduced train dataset = ",X_train_reduced.shape)
print("the shape of reduced test dataset = ",X_test_reduced.shape)"""

In [ ]:
"""#on validation data as well
text_x_val_norm_correct = text_x_val.apply(normalize_text)
tf_idf_val_correct = tfidf_vectorizer.transform(text_x_val_norm_correct)
x_val_tabular_correct = preprocessor_tabular.transform(X_val)
X_val_final_correct = hstack([x_val_tabular_correct, tf_idf_val_correct])
X_val_reduced_correct = svd.transform(X_val_final_correct)"""

**Q3)**
Use the preprocessed training data X_train_reduced.

Perform hyperparameter tuning for a RandomForestClassifier using RandomizedSearchCV.

Model Setup
Initialize the model as:
RandomForestClassifier(random_state=2306)
Do not manually set n_estimators or max_depth here, as they will be tuned.
Search only over the following parameter values:

n_estimators: [50, 100, 200]
max_depth: [5, 10, 15]

Do not include any other hyperparameters in the search.

RandomizedSearchCV Configuration is given below
n_iter = 5
cv = 3
random_state = 2306
n_jobs = -1

Do not manually specify the scoring parameter (use default).
After running: randomized_search.fit(X_train_reduced, y_train)

Retrieve the best hyperparameters using: randomized_search.best_params_

Return the best value of n_estimators selected by RandomizedSearchCV.
Return only the integer value.

In [ ]:
"""from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(random_state=2306)
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15]
}
print("RandomForestClassifier initialized:", rf_model)
from sklearn.model_selection import RandomizedSearchCV

randomized_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=5,
    cv=3,
    random_state=2306,
    n_jobs=-1,
    verbose=1
)

print("Fitting RandomizedSearchCV...")
randomized_search.fit(X_train_reduced, y_train)
print("RandomizedSearchCV fitting complete.")"""

In [ ]:
"""from sklearn.metrics import f1_score
best_params = randomized_search.best_params_
best_n_estimators = best_params['n_estimators']
print('the best parameter was found to be = ',best_n_estimators)
best_rf_model = randomized_search.best_estimator_
y_train_pred_rf = best_rf_model.predict(X_train_reduced)
y_val_pred_rf = best_rf_model.predict(X_test_reduced)
macro_f1_train_rf = f1_score(y_train, y_train_pred_rf, average='macro')
macro_f1_val_rf = f1_score(y_val, y_val_pred_rf, average='macro')
print("Macro F1 Score on the training data = ",macro_f1_train_rf)
print("Macro F1 Score on the validation data = ",macro_f1_val_rf)"""

In [ ]:
"""#testing on actual dataset
test_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample_submission_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')
print(f"Shape of test_df: {test_df.shape}")
print(f"Shape of sample_submission_df: {sample_submission_df.shape}")
test_df.drop(columns=['created_date'], inplace=True)
text_x_test = test_df['comment']
test_df.drop(columns=['comment'], inplace=True)
x_test_tabular = preprocessor_tabular.transform(test_df)
print(f"Shape of x_test_tabular after preprocessing: {x_test_tabular.shape}")
text_x_test_norm = text_x_test.apply(normalize_text)
tf_idf_test = tfidf_vectorizer.transform(text_x_test_norm)
print(f"Shape of tf_idf_test after TF-IDF vectorization: {tf_idf_test.shape}")
X_test_final = hstack([x_test_tabular, tf_idf_test])
X_test_reduced = svd.transform(X_test_final)
print(f"Shape of X_test_final after combining features: {X_test_final.shape}")
print(f"Shape of X_test_reduced after SVD: {X_test_reduced.shape}")
y_test_pred = best_rf_model.predict(X_test_reduced)
y_test_pred = best_rf_model.predict(X_test_reduced)
submission = pd.DataFrame({
    'ID': sample_submission_df['ID'],
    'label': y_test_pred
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")
print("First 5 rows of submission.csv:")
display(submission.head())"""

**Q4)**
Use the same preprocessed training data (X_train_reduced, y_train).
Do not modify or recompute preprocessing.

Step 1: Train AdaBoost
Train an AdaBoostClassifier with the following exact parameters:
n_estimators = 50
random_state = 2306
Use all other parameters as default.
Note:

Do not manually set the algorithm parameter.
Use the default behavior of AdaBoostClassifier.

Fit the model on: X_train_reduced , y_train

Step 2: Extract Estimator Errors
After fitting, access: model.estimator_errors_
This returns a NumPy array of length 50 containing the error of each weak learner.

Step 3: Compute Variance
Compute the variance of the 50 estimator errors using:
np.var(model.estimator_errors_)
Important:

Use NumPy’s default variance computation.

Step 4: Final Answer
Return the variance rounded to 4 decimal places.

In [ ]:
"""from sklearn.ensemble import AdaBoostClassifier
adaboost_model = AdaBoostClassifier(n_estimators=50, random_state=2306)
adaboost_model.fit(X_train_reduced, y_train)
print("AdaBoostClassifier trained successfully.")
estimator_errors = adaboost_model.estimator_errors_
print("\nLength of estimator_errors_ = ",len(estimator_errors))
print("First 5 estimator errors = ",estimator_errors[:5])
variance_of_errors = np.var(estimator_errors)
variance_rounded = round(variance_of_errors, 4)
print("\nVariance of the 50 estimator errors = ",variance_of_errors)
print("Rounded variance = ",variance_rounded)"""

**Q5)**
Train a RandomForestClassifier(n_estimators=100, max_depth=10, random_state=2306) using the preprocessed training data (X_train_reduced, y_train).

After fitting: Extract feature_importances_.
Give the index of the feature with the maximum importance value.
Note: If multiple features share the exact same maximum importance, return the smallest index.

In [ ]:
"""from sklearn.ensemble import RandomForestClassifier
rf_model_q5 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=2306)
rf_model_q5.fit(X_train_reduced, y_train)
print("RandomForestClassifier (Q5) trained successfully.")
feature_importances = rf_model_q5.feature_importances_
max_importance_index = np.argmax(feature_importances)
print("\nFeature importances extracted. Length = ", len(feature_importances))
print("Maximum importance value = ",feature_importances[max_importance_index])
print("Index of the feature with the maximum importance value = ",max_importance_index)"""

**Q6)**
Initialize an MLPClassifier with:

hidden_layer_sizes = (128, 64, 32)
activation = "relu"
random_state = 2306

Use the preprocessed training data (X_train_reduced , y_train).
Let N be the number of input features (i.e., X_train_reduced.shape[1]).

Compute the total number of weights (excluding biases) in the network.
Only count connection weights between layers. Do not include bias terms.
The network structure is:
Input layer: N neurons
Hidden Layer 1: 128 neurons
Hidden Layer 2: 64 neurons
Hidden Layer 3: 32 neurons
Output Layer: 4 neurons

Use your actual value of N from X_train_reduced.
Return the final integer count of weights.

In [ ]:
"""N = X_train_reduced.shape[1]
print(f"Number of input features (N) = ", N)
input_layer = N
hidden_layer_1 = 128
hidden_layer_2 = 64
hidden_layer_3 = 32
output_layer = 4
weights_1 = input_layer * hidden_layer_1
weights_2 = hidden_layer_1 * hidden_layer_2
weights_3 = hidden_layer_2 * hidden_layer_3
weights_4 = hidden_layer_3 * output_layer
total_weights = weights_1 + weights_2 + weights_3 + weights_4
print("Total number of weights (excluding biases) = ",total_weights)"""

**Q7)**
Train the same MLP architecture defined in Q6 using the preprocessed training data (X_train_reduced, y_train) with the following additional parameters:

solver = "adam", max_iter = 5, batch_size = 32, random_state = 2306

Do not modify any other parameters (keep all others as default).
After training completes (i.e., after 5 iterations), access: model.loss_
This returns the final value of the loss function after the last iteration.
Return the value of model.loss_ rounded to 4 decimal places.

In [ ]:
"""from sklearn.neural_network import MLPClassifier

hidden_layer_sizes = (128, 64, 32)
activation = "relu"
random_state = 2306
solver = "adam"
max_iter = 5
batch_size = 32
mlp_model = MLPClassifier(
    hidden_layer_sizes=hidden_layer_sizes,
    activation=activation,
    solver=solver,
    max_iter=max_iter,
    batch_size=batch_size,
    random_state=random_state
)
print("MLPClassifier initialized successfully.")
print("Architecture: hidden_layer_sizes= ",hidden_layer_sizes, "activation=",activation)
print("Training parameters: solver= ",solver, "max_iter=",max_iter, "batch_size=",batch_size, "random_state=",random_state)
print("\nFitting MLPClassifier...")
mlp_model.fit(X_train_reduced, y_train)
print("MLPClassifier training complete.")
final_loss = mlp_model.loss_
final_loss_rounded = round(final_loss, 4)
print("\nFinal loss value after ",max_iter, "iterations = ",final_loss)
print("Rounded final loss value = ",final_loss_rounded)"""

**Q8)**
Train two MLPs using the preprocessed training data (X_train_reduced, y_train) with hidden_layer_sizes=(100,), max_iter=5, and random_state=2306. use default values for other parameters unless specified.
• Model A: alpha=0.0001 (Default)
• Model B: alpha=1.0 (Strong Regularization)
Calculate the Macro F1 Score for both on the training set. What is the absolute difference between both the scores? (answer upto 4 decimal places).

In [ ]:
"""from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
common_params = {
    'hidden_layer_sizes': (100,),
    'max_iter': 5,
    'random_state': 2306,
    'solver': 'adam', # Explicitly set a solver to avoid future warnings
    'activation': 'relu' # Explicitly set activation to relu based on common practice
}

# --- Model A: Default alpha (0.0001) ---
print("Training Model A (alpha=0.0001)...")
mlp_model_A = MLPClassifier(alpha=0.0001, **common_params)
mlp_model_A.fit(X_train_reduced, y_train)
y_train_pred_A = mlp_model_A.predict(X_train_reduced)
macro_f1_A = f1_score(y_train, y_train_pred_A, average='macro')
print("Model A Macro F1 Score (training set) = ", macro_f1_A)

# --- Model B: Strong Regularization (alpha=1.0) ---
print("\nTraining Model B (alpha=1.0)...")
mlp_model_B = MLPClassifier(alpha=1.0, **common_params)
mlp_model_B.fit(X_train_reduced, y_train)
y_train_pred_B = mlp_model_B.predict(X_train_reduced)
macro_f1_B = f1_score(y_train, y_train_pred_B, average='macro')
print("Model B Macro F1 Score (training set) = ", macro_f1_B)
abs_diff_f1 = np.abs(macro_f1_A - macro_f1_B)
abs_diff_f1_rounded = round(abs_diff_f1, 4)
print("\nAbsolute difference between Model A and Model B Macro F1 Scores = ",abs_diff_f1)
print("Rounded absolute difference = ",abs_diff_f1_rounded)"""

**Q9)**
Training the MLP model as per Q7, generate a confusion matrix for the validation set. What is the sum of the off-diagonal elements (representing the total number of misclassified instances) divided by the total number of samples in the validation set? (Round to 4 decimal places).

In [ ]:
"""from sklearn.metrics import confusion_matrix
y_val_pred_mlp = mlp_model.predict(X_val_reduced_correct)
conf_matrix = confusion_matrix(y_val, y_val_pred_mlp)
print("Confusion Matrix for MLPClassifier on Validation Set:")
print(conf_matrix)
sum_all_elements = np.sum(conf_matrix)
sum_diagonal_elements = np.trace(conf_matrix)
sum_off_diagonal_elements = sum_all_elements - sum_diagonal_elements
total_validation_samples = y_val.shape[0]
misclassification_ratio = sum_off_diagonal_elements / total_validation_samples
misclassification_ratio_rounded = round(misclassification_ratio, 4)
print("\nSum of off-diagonal elements = ",sum_off_diagonal_elements)
print("Total number of samples in validation set = ",total_validation_samples)
print("Ratio (misclassified / total samples) = ",misclassification_ratio)
print("Rounded ratio = ",misclassification_ratio_rounded)"""

Applying preprocessing of Milestone 4 on test data and then generating predictions

In [ ]:
def preprocessing_final():
    import pandas as pd
    from scipy.sparse import hstack

    print("Loading test dataset...")
    # 1. Load test data
    test_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv', engine='python', on_bad_lines='warn')

    # 2. Extract and clean text feature (fill NaNs with empty string to avoid errors)
    text_x_test = test_df['comment'].fillna('')
    text_x_test_norm = text_x_test.apply(normalize_text)

    # 3. Drop columns to prep for tabular transformation
    test_df = test_df.drop(columns=['created_date', 'comment'])

    print("Transforming tabular and text features...")
    # 4. Transform using the pre-fitted global transformers (No .fit() used here)
    x_test_tabular = preprocessor_tabular.transform(test_df)
    tf_idf_test = tfidf_vectorizer.transform(text_x_test_norm)

    # 5. Combine features
    X_test_final = hstack([x_test_tabular, tf_idf_test])

    print("Applying SVD...")
    # 6. Reduce dimensionality using the pre-fitted global SVD
    X_test_reduced = svd.transform(X_test_final)

    print(f"Shape of X_test_reduced after SVD: {X_test_reduced.shape}")

    # 7. RETURN the processed data so the evaluation function can use it!
    return X_test_reduced


def model_evaluation(model, processed_test_data):
    import pandas as pd

    sample_submission_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

    print("Generating predictions...")
    # Make predictions using the data passed into the function (102,000 rows)
    y_test_pred = model.predict(processed_test_data)

    submission = pd.DataFrame({
        'ID': sample_submission_df['ID'],
        'label': y_test_pred
    })

    submission.to_csv('submission.csv', index=False)
    print("Predictions made successfully. 'submission.csv' generated.")
    display(submission.head())



In [ ]:
#final_test_data = preprocessing_final()
#model_evaluation(mlp_model_A, final_test_data)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neural_network import MLPClassifier

"""# 1. Define the model
mlp = MLPClassifier(max_iter=20, random_state=2306) # Kept max_iter low for speed while searching

# 2. Define the parameter space to search
param_dist_mlp = {
    'hidden_layer_sizes': [(100,), (128, 64), (256, 128, 64)],
    'alpha': [0.0001, 0.01, 1.0],
    'activation': ['relu', 'tanh'],
    'learning_rate_init': [0.001, 0.01]
}

# 3. Setup the search
mlp_search = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=param_dist_mlp,
    n_iter=10,        # Try 10 random combinations
    cv=3,             # 3-fold cross-validation
    scoring='f1_macro', # Optimize for the competition metric!
    n_jobs=-1,
    random_state=2306
)

#4. Fit it (this will take time!)
mlp_search.fit(X_train_reduced, y_train)
print("Best params:", mlp_search.best_params_)"""

In [ ]:
"""#print("The best parameters found were:", mlp_search.best_params_)
print("Training Model with best parameters")
#mlp_best_para = mlp_search.best_estimator_
mlp_best_para = MLPClassifier(learning_rate_init=0.001, hidden_layer_sizes=(100,), alpha=0.01, activation='relu',max_iter=2000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,      #best parameters
    tol=1e-4,)
mlp_best_para.fit(X_train_reduced, y_train)
y_train_pred_bp = mlp_best_para.predict(X_train_reduced)
macro_f1_best_para = f1_score(y_train, y_train_pred_A, average='macro')
print("Model A Macro F1 Score (training set) = ", macro_f1_best_para)"""

In [ ]:
"""final_test_data = preprocessing_final()
model_evaluation(mlp_best_para, final_test_data)"""

The MLP with parameters learning_rate_init=0.001, hidden_layer_sizes=(100,), alpha=0.01, activation='relu',max_iter=2000,  
    early_stopping=True,         
    validation_fraction=0.1,     
    n_iter_no_change=10,                                 
    tol=1e-4,)
is found to give the final test dataset score of 0.73537

**Implementing newer preprocessing :**

In [ ]:
"""def preprocessing_complete():
    import pandas as pd
    import numpy as np
    import re
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD

    # 1. Define the Text Cleaning Function
    def normalize_text(text):
        text = str(text)
        text = re.sub(r'http\S+|www\S+', '', text) # Remove URLs
        text = re.sub(r'[^a-zA-Z\s]', '', text)    # Remove punctuation/numbers
        text = re.sub(r'\s+', ' ', text).strip()   # Remove extra spaces
        return text.lower()                        # Lowercase

    # 2. Define the Feature Engineering Function
    def engineer_features(df):
        df = df.copy()

        # Text Meta-Features (Do this before cleaning the text)
        df['comment'] = df['comment'].fillna('')
        df['comment_length'] = df['comment'].apply(len)
        df['word_count'] = df['comment'].apply(lambda x: len(str(x).split()))

        # Apply text normalization
        df['comment'] = df['comment'].apply(normalize_text)

        # Datetime Features
        df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
        df['hour'] = df['created_date'].dt.hour.fillna(0)
        df['day_of_week'] = df['created_date'].dt.dayofweek.fillna(0)
        df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
        df = df.drop(columns=['created_date']) # Drop original date column

        # Log Transform highly skewed numerical columns to handle massive outliers
        skewed_cols = ['upvote', 'downvote', 'if_1', 'if_2']
        for col in skewed_cols:
            df[col] = np.log1p(df[col]) # log(1 + x)

        return df

    print("Loading datasets...")
    # 3. Load the Data
    train_df_raw = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
    test_df_raw = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')

    print("Engineering features...")
    # 4. Apply Feature Engineering
    train_df = engineer_features(train_df_raw)
    test_df = engineer_features(test_df_raw)

    # 5. Split Training Data (60/40 split with stratification)
    X = train_df.drop(columns=['label'])
    y = train_df['label']
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=2306, stratify=y)

    print("Building transformation pipeline...")
    # 6. Define Feature Groups
    numeric_features = [
        'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
        'upvote', 'downvote', 'if_1', 'if_2',
        'hour', 'day_of_week', 'is_weekend', 'comment_length', 'word_count'
    ]
    categorical_features = ['race', 'religion', 'gender', 'disability']
    text_feature = 'comment' # ColumnTransformer handles this directly!

    # 7. Create Transformers
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
    ])

    # Upgraded TF-IDF with n-grams
    text_transformer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))

    # 8. Combine into a Single Preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
            ('text', text_transformer, text_feature) # TF-IDF applied directly to the text column
        ],
        remainder='drop'
    )

    # 9. The Ultimate Pipeline (Preprocessing + SVD)
    final_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('svd', TruncatedSVD(n_components=300, random_state=2306))
    ])

    print("Fitting pipeline to training data and transforming all sets...")
    # 10. Fit ONCE on training data, Transform everywhere
    X_train_reduced = final_pipeline.fit_transform(X_train)
    X_val_reduced = final_pipeline.transform(X_val)
    X_test_reduced = final_pipeline.transform(test_df)

    print(f"Final shape of X_train_reduced: {X_train_reduced.shape}")
    print(f"Final shape of X_val_reduced: {X_val_reduced.shape}")
    print(f"Final shape of X_test_reduced: {X_test_reduced.shape}")
    print("Data is ready for modeling!")
    return X_train_reduced, X_val_reduced, X_test_reduced, y_train, y_val"""

In [ ]:
"""X_train_reduced, X_val_reduced, X_test_reduced, y_train, y_val = preprocessing_complete()
print(X_train_reduced)"""

In [ ]:
"""from sklearn.model_selection import RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
import pandas as pd

print("Setting up the MLP and RandomizedSearchCV...")

# 1. Define the base model with Early Stopping
# We set max_iter high, but early_stopping=True ensures it stops when it stops learning.
mlp_base = MLPClassifier(
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=2306
)

# 2. Define the parameter space to explore
param_dist = {
    'hidden_layer_sizes': [(100,), (128, 64), (256, 128, 64)],
    'alpha': [0.0001, 0.001, 0.01, 0.1],  # Testing different regularization strengths
    'activation': ['relu', 'tanh'],
    'learning_rate_init': [0.001, 0.005, 0.01]
}

# 3. Configure the search
mlp_search = RandomizedSearchCV(
    estimator=mlp_base,
    param_distributions=param_dist,
    n_iter=10,           # Try 10 random combinations
    cv=3,                # 3-fold cross-validation
    scoring='f1_macro',  # Target metric for the competition
    n_jobs=-1,           # Use all available CPU cores!
    random_state=2306,
    verbose=2            # Prints progress updates to your Kaggle console
)

print("Running hyperparameter search... (This might take a few minutes)")
# 4. Fit the search on the enriched training data
mlp_search.fit(X_train_reduced, y_train)

# 5. Extract the winning model

best_mlp = mlp_search.best_estimator_

print("\nSearch Complete!")

print(f"Best Parameters Found: {mlp_search.best_params_}")


# 6. Evaluate on the Validation Set
print("\nEvaluating on Validation Set...")
y_val_pred = best_mlp.predict(X_val_reduced)
val_f1 = f1_score(y_val, y_val_pred, average='macro')
print(f"Optimized Validation Macro F1 Score: {val_f1:.4f}")

# 7. Generate Final Predictions for Submission
print("\nGenerating test predictions for submission...")
test_predictions = best_mlp.predict(X_test_reduced)

# Load the sample submission to get the correct 'ID' column
sample_sub = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# Create final dataframe
final_submission = pd.DataFrame({
    'ID': sample_sub['ID'],
    'label': test_predictions
})

# Save to the root Kaggle working directory
final_submission.to_csv('submission.csv', index=False)
print("Success! 'submission.csv' is saved and ready for the leaderboard.")"""

MLP with the parameters {'learning_rate_init': 0.001, 'hidden_layer_sizes': (256, 128, 64), 'alpha': 0.0001, 'activation': 'relu'} gives a score of 0.73611

# Trying RandomForestClassifier :


In [ ]:
"""from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score
import pandas as pd

print("Setting up the RandomForest and RandomizedSearchCV...")

# 1. Define the base model
# We set n_jobs=-1 here too so the individual trees build in parallel
rf_base = RandomForestClassifier(random_state=2306, class_weight='balanced', n_jobs=-1)

# 2. Define the parameter space to explore
# These are the highest-impact parameters for controlling Random Forest behavior
param_dist_rf = {
    'n_estimators': [50, 100, 200, 300],        # Number of trees in the forest
    'max_depth': [5, 10, 15, 20, None],         # Maximum depth of each tree (None means they grow fully)
    'min_samples_split': [2, 5, 10],            # Minimum samples required to split an internal node
    'min_samples_leaf': [1, 2, 4]               # Minimum samples required to be at a leaf node
}

# 3. Configure the search
rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist_rf,
    n_iter=10,           # Try 10 random combinations
    cv=3,                # 3-fold cross-validation
    scoring='f1_macro',  # Target metric
    n_jobs=-1,           # Parallelize the search across folds
    random_state=2306,
    verbose=2            # Show progress in the console
)

print("Running hyperparameter search... (This will take a few minutes)")
# 4. Fit the search on the reduced training data
rf_search.fit(X_train_reduced, y_train)

# 5. Extract the winning model
best_rf = rf_search.best_estimator_
print("\nSearch Complete!")
print(f"Best Parameters Found: {rf_search.best_params_}")

# 6. Evaluate on the Validation Set
print("\nEvaluating on Validation Set...")
y_val_pred_rf = best_rf.predict(X_val_reduced)
val_f1_rf = f1_score(y_val, y_val_pred_rf, average='macro')
print(f"Optimized Validation Macro F1 Score: {val_f1_rf:.4f}")

# 7. Generate Final Predictions for Submission
print("\nGenerating test predictions for submission...")
test_predictions_rf = best_rf.predict(X_test_reduced)

# Load the sample submission to align the 'ID' column
sample_sub = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# Create final dataframe
final_submission_rf = pd.DataFrame({
    'ID': sample_sub['ID'],
    'label': test_predictions_rf
})

# Save to the root Kaggle working directory
final_submission_rf.to_csv('submission.csv', index=False)
print("Success! 'submission.csv' is saved and ready to be submitted.")"""

# Trying the ultimate strategy here , remember to comment this out

In [ ]:
"""import pandas as pd
import re
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Loading data...")
train_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample_sub = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# 1. Text Normalization Function
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text) # Remove special chars but keep numbers/letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

print("Cleaning text...")
train_df['comment'] = train_df['comment'].apply(normalize_text)
test_df['comment'] = test_df['comment'].apply(normalize_text)

# 2. Separate Features and Target
y_train_full = train_df['label']
X_train_full = train_df.drop(columns=['label', 'created_date'])
X_test_full = test_df.drop(columns=['created_date'])

# 3. Tabular Preprocessing
categorical_cols = ["race", "religion", "gender", "disability"]
numerical_cols = ["post_id", "emoticon_1", "emoticon_2", "emoticon_3", "upvote", "downvote", "if_1", "if_2"]

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="constant", fill_value="none")),
    ('onehot', OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_tabular = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

print("Processing Tabular Data...")
X_train_tab = preprocessor_tabular.fit_transform(X_train_full)
X_test_tab = preprocessor_tabular.transform(X_test_full)

# 4. Advanced TF-IDF Vectorization
print("Processing Text Data (TF-IDF)...")
# Upgrade: Include bigrams (e.g., "not good") and increase max_features
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=25000,
    sublinear_tf=True # logarithmic frequency (prevents extreme word counts from dominating)
)

X_train_text = tfidf_vectorizer.fit_transform(X_train_full['comment'])
X_test_text = tfidf_vectorizer.transform(X_test_full['comment'])

# 5. Combine Tabular and Text Data (Keep it Sparse!)
print("Combining Features...")
X_train_final = hstack([X_train_tab, X_train_text])
X_test_final = hstack([X_test_tab, X_test_text])

# --- Local Validation (Optional but Recommended) ---
X_tr, X_val, y_tr, y_val = train_test_split(X_train_final, y_train_full, test_size=0.2, random_state=2306, stratify=y_train_full)

print("Training Logistic Regression Model...")
# C=2.0 relaxes regularization slightly to allow the model to learn from the 25k features
# class_weight='balanced' handles your heavily imbalanced labels (0, 1, 2, 3)
model = LogisticRegression(C=2.0, max_iter=2000, class_weight='balanced', random_state=2306, n_jobs=-1)
model.fit(X_tr, y_tr)

print("\n--- Validation Results ---")
val_preds = model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")
print(classification_report(y_val, val_preds))

# --- Final Training and Submission ---
print("Retraining on entire dataset for maximum performance...")
model.fit(X_train_final, y_train_full)

print("Predicting on Test Set...")
test_predictions = model.predict(X_test_final)

# Create Submission File
submission = pd.DataFrame({
    'ID': sample_sub['ID'],
    'label': test_predictions
})

submission.to_csv('submission.csv', index=False)
print("Saved to 'submission_v2_logreg.csv'. You are ready to submit!")"""

Loading data...
Cleaning text...
Processing Tabular Data...
Processing Text Data (TF-IDF)...
Combining Features...
Training Logistic Regression Model...

--- Validation Results ---
Validation Accuracy: 0.8948
              precision    recall  f1-score   support

           0       0.98      0.92      0.95     22835
           1       0.68      0.84      0.75      3183
           2       0.87      0.87      0.87     12488
           3       0.48      0.74      0.58      1094

    accuracy                           0.89     39600
   macro avg       0.75      0.84      0.79     39600
weighted avg       0.91      0.89      0.90     39600

Retraining on entire dataset for maximum performance...
Predicting on Test Set...
Saved to 'submission.csv'

The final scoe for logistic regression model with parameters C=2.0, max_iter=2000, class_weight='balanced', random_state=2306, n_jobs=-1
is : 0.79010

Hyperparameter tuning for Logistic regression

In [ ]:
"""import pandas as pd
import re
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Loading data...")
train_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample_sub = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# 1. Text Normalization Function
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text) # Remove special chars but keep numbers/letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

print("Cleaning text...")
train_df['comment'] = train_df['comment'].apply(normalize_text)
test_df['comment'] = test_df['comment'].apply(normalize_text)

# 2. Separate Features and Target
y_train_full = train_df['label']
X_train_full = train_df.drop(columns=['label', 'created_date'])
X_test_full = test_df.drop(columns=['created_date'])

# 3. Tabular Preprocessing
categorical_cols = ["race", "religion", "gender", "disability"]
numerical_cols = ["post_id", "emoticon_1", "emoticon_2", "emoticon_3", "upvote", "downvote", "if_1", "if_2"]

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="constant", fill_value="none")),
    ('onehot', OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_tabular = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

print("Processing Tabular Data...")
X_train_tab = preprocessor_tabular.fit_transform(X_train_full)
X_test_tab = preprocessor_tabular.transform(X_test_full)

# 4. Advanced TF-IDF Vectorization
print("Processing Text Data (TF-IDF)...")
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=25000,
    sublinear_tf=True
)

X_train_text = tfidf_vectorizer.fit_transform(X_train_full['comment'])
X_test_text = tfidf_vectorizer.transform(X_test_full['comment'])

# 5. Combine Tabular and Text Data
print("Combining Features...")
X_train_final = hstack([X_train_tab, X_train_text])
X_test_final = hstack([X_test_tab, X_test_text])

# Split off a validation set for local testing
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final, y_train_full, test_size=0.2, random_state=2306, stratify=y_train_full
)

# 6. Hyperparameter Tuning for Logistic Regression
print("Starting Hyperparameter Tuning...")

# Define the base model
logreg = LogisticRegression(
    class_weight='balanced',
    random_state=2306,
    solver='saga',
    n_jobs=-1
)

# Define the parameter grid to search over

param_distributions = {
    'C': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],  # Regularization strength
    'max_iter': [1000, 2000]               # Allow more iterations for convergence
}

param_distributions = {
    'C': [0.5, 0.75, 1.0, 1.25, 1.5, 1.75],  # Regularization strength
    'max_iter': [2000, 5000]               # Allow more iterations for convergence
}
param_distributions = {
    'C': [1, 1.2, 1.3, 1.5, 1.6, 1.7],  # Regularization strength
    'max_iter': [5000, 6000]               # Allow more iterations for convergence
}
param_distributions = {
    'C': [ 0.9, 1.0, 1.1,1.05],  # Regularization strength
    'max_iter': [1000, 2000, 500,1500]               # Allow more iterations for convergence
}
# Setup RandomizedSearchCV
# Note: cv=3 means it will do 3-fold cross-validation
# n_iter=5 means it will randomly try 5 combinations from the grid above
random_search = RandomizedSearchCV(
    estimator=logreg,
    param_distributions=param_distributions,
    n_iter=6, #was 5 initially
    cv=3,
    scoring='accuracy',
    random_state=2306,
    verbose=2,
    n_jobs=-1
)

# Fit the random search on the training split
random_search.fit(X_tr, y_tr)

print(f"\nBest Hyperparameters Found: {random_search.best_params_}")

# 7. Evaluate the best model on the validation set
best_model = random_search.best_estimator_
val_preds = best_model.predict(X_val)

print("\n--- Validation Results (Best Model) ---")
print(f"Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")
print(classification_report(y_val, val_preds))

# 8. Retrain on the ENTIRE dataset using the best parameters
print("\nRetraining the best model on the complete dataset for final submission...")
# We use best_params_ to initialize a fresh model
final_model = LogisticRegression(
    **random_search.best_params_,
    #class_weight='balanced',
    random_state=2306,
    solver='saga',
    n_jobs=-1
)
final_model.fit(X_train_final, y_train_full)

print("Predicting on Test Set...")
test_predictions = final_model.predict(X_test_final)

# 9. Create Submission File
submission = pd.DataFrame({
    'ID': sample_sub['ID'],
    'label': test_predictions
})

submission.to_csv('submission.csv', index=False)
print("Saved to 'submission.csv'.")"""

1)Initially for the first guess, The hyperparameter space was made of
param_distributions = {
    'C': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],  
    'max_iter': [1000, 2000]               
}

which gave the best parameters as:
Best Hyperparameters Found: {'max_iter': 2000, 'C': 1.0} and
--- Validation Results (Best Model) ---
Validation Accuracy: 0.8956
              precision    recall  f1-score   support

           0       0.98      0.93      0.95     22835
           1       0.69      0.84      0.76      3183
           2       0.88      0.86      0.87     12488
           3       0.47      0.77      0.59      1094

    accuracy                           0.90     39600
   macro avg       0.75      0.85      0.79     39600
weighted avg       0.91      0.90      0.90     39600


This Gave a score of ***0.79247***


2)Now narrowing the search with parameter space : param_distributions = {
    'C': [0.5, 0.75, 1.0, 1.25, 1.5, 1.75],  
    'max_iter': [2000, 5000]               
}
Best Hyperparameters Found: {'max_iter': 5000, 'C': 1.5}
	--- Validation Results (Best Model) ---
	Validation Accuracy: 0.8956
	              precision    recall  f1-score   support

	           0       0.98      0.93      0.95     22835
	           1       0.68      0.84      0.76      3183
	           2       0.87      0.87      0.87     12488
	           3       0.48      0.75      0.58      1094

	    accuracy                           0.90     39600
	   macro avg       0.75      0.85      0.79     39600
	weighted avg       0.91      0.90      0.90     39600




and class weight is not set to balanced gives a score of ***0.78900***
3)with parameters : {'max_iter': 6000, 'C': 1.3}
--- Validation Results (Best Model) ---
	Validation Accuracy: 0.8957
	              precision    recall  f1-score   support

	           0       0.98      0.93      0.95     22835
	           1       0.69      0.84      0.76      3183
	           2       0.88      0.86      0.87     12488
	           3       0.47      0.76      0.58      1094

	    accuracy                           0.90     39600
	   macro avg       0.75      0.85      0.79     39600
	weighted avg       0.91      0.90      0.90     39600

we get a score of ***0.78637***

3) for parameters {'max_iter': 2000, 'C': 1.1} ,
    --- Validation Results (Best Model) ---
Validation Accuracy: 0.8966
              precision    recall  f1-score   support

           0       0.98      0.93      0.95     22835
           1       0.69      0.84      0.76      3183
           2       0.88      0.86      0.87     12488
           3       0.49      0.77      0.60      1094

    accuracy                           0.90     39600
   macro avg       0.76      0.85      0.79     39600
weighted avg       0.91      0.90      0.90     39600

and we get score : 0.77287

# Now trying the LGBMClassifier comment this out as well

In [ ]:
import pandas as pd
import re
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from lightgbm import LGBMClassifier

print("Loading data...")
train_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample_sub = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# 1. Text Normalization Function
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text) # Remove special chars but keep numbers/letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

print("Cleaning text...")
train_df['comment'] = train_df['comment'].apply(normalize_text)
test_df['comment'] = test_df['comment'].apply(normalize_text)

# 2. Separate Features and Target
y_train_full = train_df['label']
X_train_full = train_df.drop(columns=['label', 'created_date'])
X_test_full = test_df.drop(columns=['created_date'])

# 3. Tabular Preprocessing
categorical_cols = ["race", "religion", "gender", "disability"]
numerical_cols = ["post_id", "emoticon_1", "emoticon_2", "emoticon_3", "upvote", "downvote", "if_1", "if_2"]

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="constant", fill_value="none")),
    ('onehot', OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_tabular = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

print("Processing Tabular Data...")
X_train_tab = preprocessor_tabular.fit_transform(X_train_full)
X_test_tab = preprocessor_tabular.transform(X_test_full)

# 4. Advanced TF-IDF Vectorization
print("Processing Text Data (TF-IDF)...")
# Upgrade: Include bigrams (e.g., "not good") and increase max_features
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=25000,
    sublinear_tf=True # logarithmic frequency (prevents extreme word counts from dominating)
)

X_train_text = tfidf_vectorizer.fit_transform(X_train_full['comment'])
X_test_text = tfidf_vectorizer.transform(X_test_full['comment'])

# 5. Combine Tabular and Text Data (Keep it Sparse!)
print("Combining Features...")
X_train_final = hstack([X_train_tab, X_train_text])
X_test_final = hstack([X_test_tab, X_test_text])

# --- Local Validation (Optional but Recommended) ---
X_tr, X_val, y_tr, y_val = train_test_split(X_train_final, y_train_full, test_size=0.2, random_state=2306, stratify=y_train_full)

print("Training LGBM classifier Model...")
# C=2.0 relaxes regularization slightly to allow the model to learn from the 25k features
# class_weight='balanced' handles your heavily imbalanced labels (0, 1, 2, 3)

#comment this below older model out
"""model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    #class_weight='balanced',
    random_state=2306,
    n_jobs=-1,
    verbose=1
)"""
"""model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=16,
    class_weight='balanced',
    random_state=2306,
    n_jobs=-1,
    verbose=1
)"""
model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=16,
    #class_weight='balanced',
    random_state=2306,
    n_jobs=-1,
    verbose=1
)
#new parameters
"""model = LGBMClassifier(
    n_estimators=1200,          # More trees
    learning_rate=0.03,         # Slower learning rate to pair with more trees
    num_leaves=64,              # Let the tree find more complex interactions (default is 31)
    max_depth=-1,               # Let num_leaves control the depth
    min_child_samples=30,       # Prevent overfitting on very rare words
    subsample=0.8,              # Row sampling
    colsample_bytree=0.6,       # Only use 60% of the 25k features per tree (huge for NLP)
    random_state=2306,
    n_jobs=-1,
    verbose=1,
    # class_weight='balanced'   # <-- TRY COMMENTING THIS OUT FIRST
)"""
model.fit(X_tr, y_tr)

print("\n--- Validation Results ---")
val_preds = model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")
print(classification_report(y_val, val_preds))

# --- Final Training and Submission ---
print("Retraining on entire dataset for maximum performance...")
model.fit(X_train_final, y_train_full)

print("Predicting on Test Set...")
test_predictions = model.predict(X_test_final)

# Create Submission File
submission = pd.DataFrame({
    'ID': sample_sub['ID'],
    'label': test_predictions
})

submission.to_csv('submission.csv', index=False)
print("Saved to 'submission.csv'. You are ready to submit!")

1) Training the LGBMClassifier with the hyperparameters : (
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    class_weight='balanced',
    random_state=2306,
    n_jobs=-1
)

Validation Accuracy: 0.8987
              precision    recall  f1-score   support

           0       0.98      0.94      0.96     22835
           1       0.70      0.85      0.77      3183
           2       0.88      0.84      0.86     12488
           3       0.46      0.73      0.57      1094

    accuracy                           0.90     39600
   macro avg       0.75      0.84      0.79     39600
weighted avg       0.91      0.90      0.90     39600

This gives a final test score of : ***0.79589***

2)Training the LGBMClassifier with the hyperparameters : (
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=2306,
    n_jobs=-1
)

Validation Accuracy: 0.9104
              precision    recall  f1-score   support

           0       0.98      0.95      0.96     22835
           1       0.79      0.77      0.78      3183
           2       0.84      0.92      0.88     12488
           3       0.76      0.42      0.54      1094

    accuracy                           0.91     39600
   macro avg       0.84      0.76      0.79     39600
weighted avg       0.91      0.91      0.91     39600

This gives a final test score of : ***0.79026***

3)Training the LGBMClassifier with the hyperparameters : (
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=16,
    class_weight='balanced',
    random_state=2306,
    n_jobs=-1,
    verbose=1
)

Validation Accuracy: 0.9023
              precision    recall  f1-score   support

           0       0.98      0.94      0.96     22835
           1       0.70      0.86      0.77      3183
           2       0.89      0.85      0.87     12488
           3       0.48      0.74      0.58      1094

    accuracy                           0.90     39600
   macro avg       0.76      0.85      0.80     39600
weighted avg       0.91      0.90      0.91     39600

This gives a final test score of : ***0.79809***

4)Training the LGBMClassifier with the hyperparameters : (
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=16,
    random_state=2306,
    n_jobs=-1,
    verbose=1
)

Validation Accuracy: 0.9134
              precision    recall  f1-score   support

           0       0.98      0.95      0.96     22835
           1       0.79      0.78      0.79      3183
           2       0.85      0.92      0.88     12488
           3       0.76      0.48      0.59      1094

    accuracy                           0.91     39600
   macro avg       0.84      0.78      0.81     39600
weighted avg       0.91      0.91      0.91     39600

This gives a final test score of : ***0.80636***

5)Training the LGBMClassifier with the parameters : model = LGBMClassifier(
    n_estimators=1200,          
    learning_rate=0.03,         
    num_leaves=64,              
    max_depth=-1,               
    min_child_samples=30,       
    subsample=0.8,              
    colsample_bytree=0.6,       
    random_state=2306,
    n_jobs=-1,
    verbose=1,
    )

    Validation Accuracy: 0.9129
              precision    recall  f1-score   support

           0       0.97      0.95      0.96     22835
           1       0.79      0.78      0.79      3183
           2       0.85      0.92      0.88     12488
           3       0.74      0.52      0.61      1094

    accuracy                           0.91     39600
   macro avg       0.84      0.79      0.81     39600
weighted avg       0.91      0.91      0.91     39600

    
Gave a final test score of : ***0.81315***